In [6]:
import requests
import pandas as pd
import os
from datetime import datetime

os.makedirs('data/raw', exist_ok=True)

print("=" * 80)
print("DOWNLOADING WORLD BANK DATA VIA API")
print("=" * 80)

# Define humanitarian countries
humanitarian_countries = {
    'KEN': 'Kenya',
    'UGA': 'Uganda',
    'ETH': 'Ethiopia',
    'SOM': 'Somalia',
    'YEM': 'Yemen',
    'MMR': 'Myanmar',
    'AFG': 'Afghanistan',
    'SYR': 'Syrian Arab Republic',
    'COD': 'Congo, Dem. Rep.',
    'PAK': 'Pakistan',
    'NGA': 'Nigeria',
    'BDI': 'Burundi',
    'RWA': 'Rwanda',
    'MOZ': 'Mozambique',
    'ZWE': 'Zimbabwe',
}

# Define health indicators
indicators = {
    'SP.DYN.CDRT.IN': 'Mortality rate, under-5 (per 1000 live births)',
    'SP.DYN.LE00.IN': 'Life expectancy at birth (years)',
    'SH.MMR.MORT.NE': 'Maternal mortality ratio (per 100000 live births)',
    'SH.TBS.INCD': 'Incidence of tuberculosis (per 100000)',
    'SH.XPD.CHEX': 'Health expenditure per capita (current US dollar)',
    'SH.IMM.DPPT': 'Immunization, DPT (percentage of children ages 12-23 months)',
    'SH.STA.ACSN': 'Improved sanitation facilities (percentage of population with access)',
}

print(f"\nCountries: {len(humanitarian_countries)}")
print(f"Indicators: {len(indicators)}")
print(f"Time period: 2010-2022")
print(f"\nDownloading data from World Bank API...\n")

all_data = []

# Loop through each indicator
for idx, (indicator_code, indicator_name) in enumerate(indicators.items(), 1):
    print(f"[{idx}/{len(indicators)}] Downloading {indicator_code}...", end=" ")
    
    try:
        # Build API URL for this indicator
        country_codes = ';'.join(humanitarian_countries.keys())
        url = f"https://api.worldbank.org/v2/country/{country_codes}/indicator/{indicator_code}?date=2010:2022&format=json&per_page=1000"
        
        # Fetch data
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Parse JSON response
        data = response.json()
        
        if len(data) < 2 or data[1] is None:
            print("No data")
            continue
        
        records = data[1]
        
        # Extract relevant fields
        for record in records:
            if record['value'] is not None:
                all_data.append({
                    'Country Name': record['country']['value'],
                    'Country Code': record['countryiso3code'],
                    'Indicator Name': indicator_name,
                    'Indicator Code': indicator_code,
                    'Year': record['date'],
                    'Value': record['value'],
                })
        
        print(f"OK ({len([r for r in records if r['value'] is not None])} records)")
        
    except Exception as e:
        print(f"Error: {str(e)[:50]}")

print(f"\nTotal records downloaded: {len(all_data)}")

# Convert to DataFrame
df = pd.DataFrame(all_data)

if len(df) == 0:
    print("ERROR: No data was downloaded")
    print("This might be due to:")
    print("  1. Internet connection issue")
    print("  2. World Bank API is temporarily down")
    print("  3. Invalid country or indicator codes")
    exit(1)

# Pivot to wide format (years as columns)
print("\nConverting to wide format...")

df_pivot = df.pivot_table(
    index=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    columns='Year',
    values='Value',
    aggfunc='first'
).reset_index()

# Save to CSV
output_path = 'data/raw/worldbank_health_data.csv'
df_pivot.to_csv(output_path, index=False)

print(f"\nFile saved: {output_path}")
print(f"Shape: {df_pivot.shape}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

print("\n" + "=" * 80)
print("DOWNLOAD COMPLETE")
print("=" * 80)
print(f"\nNext step: Run cleaning script")
print(f"  python 4_clean_worldbank_data.py")

DOWNLOADING WORLD BANK DATA VIA API

Countries: 15
Indicators: 7
Time period: 2010-2022


[1/7] Downloading SP.DYN.CDRT.IN... OK (195 records)
[2/7] Downloading SP.DYN.LE00.IN... OK (195 records)
[3/7] Downloading SH.MMR.MORT.NE... No data
[4/7] Downloading SH.TBS.INCD... OK (195 records)
[5/7] Downloading SH.XPD.CHEX... No data
[6/7] Downloading SH.IMM.DPPT... No data
[7/7] Downloading SH.STA.ACSN... No data

Total records downloaded: 585

Converting to wide format...

File saved: data/raw/worldbank_health_data.csv
Shape: (45, 17)
File size: 6.8 KB

DOWNLOAD COMPLETE

Next step: Run cleaning script
  python 4_clean_worldbank_data.py


# Step 2: Clean the Downloaded Data

In [7]:
import pandas as pd
import os

os.makedirs('data/cleaned', exist_ok=True)

print("=" * 80)
print("CLEANING WORLD BANK DATA")
print("=" * 80)

# Load the downloaded CSV file
print("\n[1] Loading World Bank data...")
df_wb = pd.read_csv('data/raw/worldbank_health_data.csv')

print(f"   Loaded shape: {df_wb.shape}")
print(f"   Columns available: {df_wb.columns.tolist()[:10]}")

# Inspect data
print("\n[2] Inspecting data...")
print(f"\nFirst 2 rows:")
print(df_wb.head(2))

print(f"\nIndicators found:")
print(df_wb['Indicator Name'].unique())

# Reshape from wide to long format
print("\n[3] Reshaping data (wide to long format)...")

year_columns = [str(year) for year in range(2010, 2023)]

# Keep only available year columns
year_columns = [col for col in year_columns if col in df_wb.columns]

df_clean = df_wb[['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'] + year_columns]

# Rename for clarity
df_clean = df_clean.rename(columns={
    'Country Name': 'country_name',
    'Country Code': 'country_code',
    'Indicator Name': 'indicator_name',
    'Indicator Code': 'indicator_code'
})

# Melt to long format
df_long = df_clean.melt(
    id_vars=['country_name', 'country_code', 'indicator_name', 'indicator_code'],
    var_name='year',
    value_name='value'
)

# Clean data
df_long['year'] = pd.to_numeric(df_long['year'], errors='coerce')
df_long = df_long.dropna(subset=['year'])
df_long['year'] = df_long['year'].astype(int)
df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

print(f"   Reshaped to: {df_long.shape}")

# Create pivot table with indicators as columns
print("\n[4] Creating wide format for all indicators...")

# Map indicator codes to column names
indicator_map = {
    'SP.DYN.CDRT.IN': 'under_5_mortality_rate',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SH.MMR.MORT.NE': 'maternal_mortality_ratio',
    'SH.TBS.INCD': 'tuberculosis_incidence',
    'SH.XPD.CHEX': 'health_exp_per_capita_usd',
    'SH.IMM.DPPT': 'dpt_immunization_pct',
    'SH.STA.ACSN': 'improved_sanitation_access_pct',
}

# Create master table
df_master = df_long[['country_name', 'country_code', 'year']].drop_duplicates()

# Add each available indicator
for indicator_code, column_name in indicator_map.items():
    df_indicator = df_long[df_long['indicator_code'] == indicator_code].copy()
    
    if len(df_indicator) > 0:
        df_indicator = df_indicator[['country_code', 'year', 'value']].rename(
            columns={'value': column_name}
        )
        
        df_master = df_master.merge(
            df_indicator,
            on=['country_code', 'year'],
            how='left'
        )
        
        print(f"   Added {column_name}")
    else:
        print(f"   Skipped {column_name} (no data)")

# Add region information
print("\n[5] Adding region metadata...")

region_map = {
    'Kenya': 'East Africa',
    'Uganda': 'East Africa',
    'Ethiopia': 'East Africa',
    'Somalia': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Mozambique': 'Southern Africa',
    'Zimbabwe': 'Southern Africa',
    'Nigeria': 'West Africa',
    'Myanmar': 'Southeast Asia',
    'Pakistan': 'South Asia',
    'Yemen': 'Middle East',
    'Afghanistan': 'South Asia',
    'Syrian Arab Republic': 'Middle East',
    'Congo, Dem. Rep.': 'Sub-Saharan Africa',
}

df_master['region'] = df_master['country_name'].map(region_map)

print(f"   Added region info")

# Sort and save
print("\n[6] Finalizing and saving...")

df_master = df_master.sort_values(['region', 'country_name', 'year']).reset_index(drop=True)

output_path = 'data/cleaned/worldbank_humanitarian_health_2010_2022.csv'
df_master.to_csv(output_path, index=False)

print()
print("=" * 80)
print("DATA CLEANING COMPLETE")
print("=" * 80)

print(f"\nDataset: {output_path}")
print(f"Shape: {df_master.shape}")
print(f"\nColumns:")
for col in df_master.columns:
    print(f"  {col}")

print(f"\nPreview (first 10 rows):")
print(df_master.head(10).to_string())

print(f"\nMissing data by column:")
print(df_master.isnull().sum())

print(f"\nReady to load into MySQL")

CLEANING WORLD BANK DATA

[1] Loading World Bank data...
   Loaded shape: (45, 17)
   Columns available: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '2010', '2011', '2012', '2013', '2014', '2015']

[2] Inspecting data...

First 2 rows:
  Country Name Country Code                          Indicator Name  \
0  Afghanistan          AFG  Incidence of tuberculosis (per 100000)   
1  Afghanistan          AFG        Life expectancy at birth (years)   

   Indicator Code     2010    2011     2012     2013    2014    2015     2016  \
0     SH.TBS.INCD  218.000  204.00  194.000  194.000  197.00  200.00  204.000   
1  SP.DYN.LE00.IN   60.702   61.25   61.735   62.188   62.26   62.27   62.646   

      2017     2018     2019     2020     2021     2022  
0  209.000  212.000  213.000  205.000  206.000  206.000  
1   62.406   62.443   62.941   61.454   60.417   65.617  

Indicators found:
['Incidence of tuberculosis (per 100000)'
 'Life expectancy at birth (years)'
 'Mortalit